# DI-VNN Model Training and Evaluation

This notebook trains Deep-Insight Visible Neural Network (DI-VNN) models for preeclampsia pathogenesis prediction.

The training pipeline:
1. Seed selection (10 random seeds, pick best)
2. L2-norm tuning (0.01, 0.1, 1, 10)
3. Final training with best seed and L2
4. Prediction on training, validation, and test sets
5. Grad-CAM for explainability

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.utils import resample
# from sklearn.model_selection import BaseCrossValidator
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from joblib import dump, parallel_backend, load
import os
# os.environ['CUDA_LAUNCH_BLOCKING']='1'
import matplotlib.pyplot as plt

import math
import regex as re
from divnn import utils, TidySet
from divnn.ExpressionSet import *
from dfply import X, mutate, mask, select, rename, left_join
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
from torch.nn.parameter import Parameter
from torch.utils.data import random_split
from torch.utils.data import Subset
import torch.optim as optim
from progressbar import ProgressBar
from tqdm.notebook import tqdm
# import gc

import shap
#from lime.lime_tabular import LimeTabularExplainer

In [ ]:
from utils import ontonet, OntoDataset, train_model, best_configuration, model_predict, model_grad_cam

## 3. Setup: File Lists and Parameters

In [ ]:
# List of training tidy set file names
training_tidy_set_files=[
    'data/divnn_models/pred_data_training_cutoff_03.ts.tar.gz'
    ,'data/divnn_models/pred_data_training_cutoff_06.ts.tar.gz'
    ,'data/divnn_models/pred_data_training_cutoff_09.ts.tar.gz'
    ,'data/divnn_models/pred_data_training_cutoff_12.ts.tar.gz'
    ,'data/divnn_models/pred_data_training_cutoff_no.ts.tar.gz'
]

# List of tidy set files to make predictions
prediction_tidy_set_files={}
for training_tidy_set_file in training_tidy_set_files:
    filename_prefix=os.path.basename(training_tidy_set_file).split('.')[0]
    prediction_tidy_set_files[filename_prefix]=[
        f'data/divnn_models/{filename_prefix}_target/pred_data_training.ts.tar.gz'
        ,f'data/divnn_models/{filename_prefix}_target/pred_data_validation.ts.tar.gz'
        ,f'data/divnn_models/{filename_prefix}_target/pred_data_test.ts.tar.gz'
    ]

## 4. Seed Selection and L2-Norm Tuning

For each model and L2-norm:
1. Try 10 random seeds with 1 epoch each
2. Select best seed using `best_configuration()`
3. Train with best seed for 30 epochs

In [ ]:
epochs=30
patience=30/5
batch_size=32
warm_up=0.05
lr=2e-6
np.random.seed(seed)
trial_seeds=np.random.permutation(np.arange(10000))[:10]
with open(r'data\divnn_models\best_seeds.csv', 'w') as csvfile:
    csvfile.write('l2_norm,training_tidy_set_file,best_seed\n')
    for l2_norm in [0.01,0.1,1,10]:
        for training_tidy_set_file in training_tidy_set_files:
            print(training_tidy_set_file)
            filename_prefix=os.path.basename(training_tidy_set_file).replace('.ts.tar.gz','')
            save_dir=f'data/divnn_models/{filename_prefix}_ontonet_tuning/l2_norm_{l2_norm}/'
            
            # Try 10 different seeds
            train_auc_rocs=[]
            val_auc_rocs=[]
            for i in range(len(trial_seeds)):
                trial_seed=int(trial_seeds[i])
                print(f"Seed: {trial_seed} (i={i+1}/{len(trial_seeds)})")
                # Run train_model with 1 epoch and 1 patience
                train_auc_roc,val_auc_roc=train_model(
                    tidy_set_path=training_tidy_set_file
                    ,save_dir=save_dir
                    ,l2_norm=l2_norm
                    ,epochs=1
                    ,patience=1
                    ,batch_size=batch_size
                    ,warm_up=warm_up
                    ,lr=lr
                    ,find_seed_mode=True
                    ,seed=trial_seed
                )
                
                train_auc_rocs.append(train_auc_roc)
                val_auc_rocs.append(val_auc_roc)
                
                # gc.collect()
                # if torch.cuda.is_available():
                #     torch.cuda.empty_cache()
            
            # Check which seed gives the best scores
            best_seed=int(trial_seeds[best_configuration(train_auc_rocs,val_auc_rocs,alpha=0.5)])
            print(f"Best seed: {best_seed}")
            
            # Run train_model again with the best seed and original epochs and patience
            train_auc_roc,val_auc_roc=train_model(
                tidy_set_path=training_tidy_set_file
                ,save_dir=save_dir
                ,l2_norm=l2_norm
                ,epochs=epochs
                ,patience=patience
                ,batch_size=batch_size
                ,warm_up=warm_up
                ,lr=lr
                ,find_seed_mode=False
                ,seed=best_seed
            )
            
            # Write the data row
            csvfile.write(f'{l2_norm},{training_tidy_set_file},{best_seed}\n')
            
            # gc.collect()
            # if torch.cuda.is_available():
            #     torch.cuda.empty_cache()

## 5. Best L2 Selection

Best L2-norm is selected in R (`index.Rmd`) by reading tuning histories.
The result is saved in `data/divnn_models/best_l2_norm.csv`.

## 6. Final Training

Train with best L2-norm for 500 epochs (patience=250).

In [ ]:
l2_norm=0.01
for training_tidy_set_file in training_tidy_set_files[1:]:
    filename_prefix=os.path.basename(training_tidy_set_file).split('.')[0]
    train_model(
        tidy_set_path=training_tidy_set_file
        ,save_dir=f'data/divnn_models/{filename_prefix}_ontonet/'
        ,l2_norm=l2_norm
        ,epochs=500
        ,patience=500/2
        ,batch_size=32
        ,warm_up=0.05
        ,lr=2e-6
        ,seed=seed
    )

## 7. Prediction on Train/Validation/Test Sets

In [ ]:
for training_tidy_set_file in training_tidy_set_files:
    
    filename_prefix=os.path.basename(training_tidy_set_file).split('.')[0]
    
    for prediction_tidy_set_file in prediction_tidy_set_files[filename_prefix]:
        model[f'{filename_prefix}_ontonet']=model_predict(
            tidy_set_path=training_tidy_set_file
            ,model_weight_dir=f'data/divnn_models/{filename_prefix}_ontonet/'
            ,l2_norm=0.01
            ,batch_size=32
            ,seed=seed
            ,new_tidy_set_path=prediction_tidy_set_file
        )